<a href="https://colab.research.google.com/github/adnan-hussen/system-one-vlm/blob/main/vlm_system_one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

adnanabdulfetah_testdata_path = kagglehub.dataset_download('adnanabdulfetah/testdata')

print('Data source import complete.')


100%|██████████| 165k/165k [00:00<00:00, 75.4MB/s]

Extracting files...
Data source import complete.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


In [5]:
!pip install --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0


In [6]:
import torch
import numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForMultimodalLM

#initialization
device = "cuda"
model_id = "Qwen/Qwen3.5-2B"

print(f"Loading {model_id} on {device}...")
processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-2B")
model = AutoModelForMultimodalLM.from_pretrained(
    "Qwen/Qwen3.5-2B",
    device_map="auto",
    dtype = 'float16'
)
model.eval()


# tokenizer
from functools import lru_cache

@lru_cache(maxsize=None)
def _vocab_index(tokenizer) -> dict[str, tuple[int, ...]]:
    #map normalized token text to token ids once
    index: dict[str, list[int]] = {}
    for tok_id in range(len(tokenizer)):
        text = tokenizer.decode([tok_id]).strip().lower()
        if text:
            index.setdefault(text, []).append(tok_id)
    return {k: tuple(v) for k, v in index.items()}

def get_label_token_ids(tokenizer, label: str) -> list[int]:
    #all single-token ids that spell the label (any case, with or without a leading space)
    ids = _vocab_index(tokenizer).get(label.strip().lower())
    if not ids:
        raise ValueError(f"No single-token form of {label!r} exists in the vocabulary.")
    return list(ids)

def last_token_logits(inputs) -> torch.Tensor:
    #forward pass returning the final position's logits
    with torch.inference_mode():
        try:
            outputs = model(**inputs, logits_to_keep=1)
        except TypeError:
            outputs = model(**inputs)
        return outputs.logits[0, -1, :].float()


# decision evaluator
def evaluate_decision(
    image: Image.Image,
    state: dict,
    question: str,
    primitive_type: str,
    criteria: dict | list,
    temperature: float = 1.0,
    calibrate_prior: bool = False
) -> dict:
    """
    Supported primitives:
      - 'noul': Binary verification P(True) in [0.0, 1.0]
      - 'choice': Mutually exclusive categorical selection
      - 'score': Continuous expectation and entropy confidence across an ordered rubric
    """
    # setup candidate target labels
    if primitive_type == "noul":
        target_labels = ["Yes", "No"]
        options_text = (
            f"Yes: {criteria.get('true', 'Condition is met')}\n"
            f"No: {criteria.get('false', 'Condition is not met')}"
        )
    elif primitive_type == "choice":
        target_labels = list(criteria.keys())
        options_text = "\n".join([f"{k}: {desc}" for k, desc in criteria.items()])
    elif primitive_type == "score":
        target_labels = [str(i) for i in range(len(criteria))]
        options_text = "\n".join([f"{i}: {desc}" for i, desc in enumerate(criteria)])
    else:
        raise ValueError(f"Unsupported primitive: {primitive_type}")

    # prompt construction
    prompt_body = (
        f"[METADATA / CONTEXT]\n{state}\n\n"
        f"[QUESTION]\n{question}\n\n"
        f"[OPTIONS]\n{options_text}\n\n"
        f"Select the exact character of your choice. Do not provide explanations."
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_body}
            ]
        }
    ]

    # chat template formatting
    base_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    formatted_text = base_text + "Choice: "

    # process inputs
    inputs = processor(
        text=[formatted_text],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    # single forward pass

    final_token_logits = last_token_logits(inputs)


    label_logits = []
    for lbl in target_labels:
        token_ids = get_label_token_ids(processor.tokenizer, lbl)
        # pool all valid tokens into a single logit using LogSumExp function
        pooled_logit = torch.logsumexp(final_token_logits[token_ids], dim=-1)
        label_logits.append(pooled_logit)

    stacked_logits = torch.stack(label_logits)

    # prior subtraction too reduce model's bias
    if calibrate_prior:
        prior_logits = compute_context_free_prior(
            target_labels,
            options_text,
            model,
            processor
        )
        stacked_logits = stacked_logits - prior_logits

    # temperature scaled softmax
    scaled_logits = stacked_logits / temperature
    probabilities = torch.softmax(scaled_logits, dim=-1).cpu().numpy()
    prob_map = {lbl: float(round(p, 4)) for lbl, p in zip(target_labels, probabilities)}

    # construct the structured decision outputs
    if primitive_type == "noul":
        return {
            "type": "noul",
            "noul": prob_map["Yes"],
            "probabilities": prob_map
        }

    elif primitive_type == "choice":
        winning_idx = int(np.argmax(probabilities))
        return {
            "type": "choice",
            "choice": target_labels[winning_idx],
            "confidence": float(round(probabilities[winning_idx], 4)),
            "probabilities": prob_map
        }

    elif primitive_type == "score":
        weights = np.arange(len(criteria))
        expected_score = float(round(np.sum(weights * probabilities), 3))

        # normalized shannon entropy confidence
        k = len(criteria)
        if k > 1:
            entropy = -np.sum(probabilities * np.log(probabilities + 1e-12))
            confidence = float(round(1.0 - (entropy / np.log(k)), 3))
        else:
            confidence = 1.0

        return {
            "type": "score",
            "score": expected_score,
            "confidence": confidence,
            "probabilities": prob_map,
            "legend": {str(i): desc for i, desc in enumerate(criteria)}
        }


# context free prior calculator
def compute_context_free_prior(
    target_labels: list[str],
    options_text: str,
    model,
    processor
) -> torch.Tensor:
    """
    Evaluates an uninformative black image to extract the model's base
    letter and frequency priors for logit subtraction.
    """
    dummy_image = Image.new("RGB", (64, 64), color="black")
    dummy_prompt = (
        f"[OPTIONS]\n{options_text}\n\n"
        f"Select the exact character of your choice. Do not provide explanations."
    )

    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": dummy_image},
            {"type": "text", "text": dummy_prompt}
        ]}
    ]

    formatted = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + "Choice: "
    inputs = processor(text=[formatted], images=[dummy_image], return_tensors="pt").to(model.device)

    terminal_logits = last_token_logits(inputs)

    prior_logits = []
    for lbl in target_labels:
        token_ids = get_label_token_ids(processor.tokenizer, lbl)
        prior_logits.append(torch.logsumexp(terminal_logits[token_ids], dim=-1))

    return torch.stack(prior_logits)

Loading Qwen/Qwen3.5-2B on cuda...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

In [11]:
#test image
test_image = Image.open('fire.jpg')

state_payload = {}

choice_res = evaluate_decision(
    image=test_image,
    state=state_payload,
    question="Who is in this image?",
    primitive_type="choice",
    criteria={
        "A": "cop",
        "B": "firefighter",
        "C": "soldier"
    },
    temperature=1.0,
    calibrate_prior=True
)
print("Choice Result:")
print(choice_res)
print("-" * 50)


noul_res = evaluate_decision(
    image=test_image,
    state=state_payload,
    question="Does the image contain fire?",
    primitive_type="noul",
    criteria={
    },
    temperature=1.4
)
print("NOUL RESULT:")
print(noul_res)
print("-" * 50)



Choice Result:
{'type': 'choice', 'choice': 'B', 'confidence': 0.9007999897003174, 'probabilities': {'A': 0.045099999755620956, 'B': 0.9007999897003174, 'C': 0.05400000140070915}}
--------------------------------------------------
NOUL RESULT:
{'type': 'noul', 'noul': 0.9613999724388123, 'probabilities': {'Yes': 0.9613999724388123, 'No': 0.038600001484155655}}
--------------------------------------------------


In [14]:
#benchmarking script

import time
import statistics
import torch
from PIL import Image


IMAGE_PATH = "fire.jpg"

# different image sizes to try
SIZES = [224, 336, 448, 672, 896, 1344, None]

N_WARMUP = 3
N_ITERS = 20

shared_state = {}

CASES = {
    "noul": dict(
        question="Is there fire visible in the image?",
        primitive_type="noul",
        criteria={"true": "There is fire.", "false": "There is no fire."},
    ),
    "choice": dict(
        question="Who is visible in the image?",
        primitive_type="choice",
        criteria={"A": "a fire fighter", "B": "a cop", "C": "a doctor"},
    ),
    "score": dict(
        question="How large is the fire?",
        primitive_type="score",
        criteria=[
            "No fire visible",
            "Small, contained flame",
            "Moderate fire",
            "Large, spreading fire",
            "Massive, out-of-control fire",
        ],
    ),
}


HAS_CUDA = torch.cuda.is_available()

def sync():
    if HAS_CUDA:
        torch.cuda.synchronize()

def resized(image: Image.Image, longest_side):
    """Downscale so the longest side equals `longest_side` (never upscales)."""
    if longest_side is None:
        return image
    img = image.copy()
    img.thumbnail((longest_side, longest_side))
    return img

def count_input_tokens(image: Image.Image, question: str, primitive_type: str, criteria) -> int:
    """Rebuild the exact prompt evaluate_decision uses and return its token length."""
    if primitive_type == "noul":
        options_text = (f"Yes: {criteria.get('true', 'Condition is met')}\n"
                        f"No: {criteria.get('false', 'Condition is not met')}")
    elif primitive_type == "choice":
        options_text = "\n".join(f"{k}: {d}" for k, d in criteria.items())
    else:
        options_text = "\n".join(f"{i}: {d}" for i, d in enumerate(criteria))

    prompt_body = (
        f"[METADATA / CONTEXT]\n{shared_state}\n\n"
        f"[QUESTION]\n{question}\n\n"
        f"[OPTIONS]\n{options_text}\n\n"
        f"Select the exact character of your choice. Do not provide explanations."
    )
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt_body},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + "Choice: "
    inputs = processor(text=[text], images=[image], return_tensors="pt")
    return int(inputs["input_ids"].shape[1])

def benchmark(fn, n_warmup=N_WARMUP, n_iters=N_ITERS) -> dict:
    """Time fn() and measure its GPU memory footprint above the resident baseline."""
    for _ in range(n_warmup):
        fn()

    if HAS_CUDA:
        sync()
        torch.cuda.empty_cache()
        baseline = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()

    times = []
    for _ in range(n_iters):
        sync()
        t0 = time.perf_counter()
        fn()
        sync()
        times.append((time.perf_counter() - t0) * 1000)
    times.sort()

    result = {
        "mean_ms": statistics.mean(times),
        "std_ms": statistics.stdev(times) if len(times) > 1 else 0.0,
        "p50_ms": times[len(times) // 2],
        "p95_ms": times[max(int(len(times) * 0.95) - 1, 0)],
        "min_ms": times[0],
        "max_ms": times[-1],
    }
    result["decisions_per_sec"] = 1000 / result["mean_ms"]

    if HAS_CUDA:
        result["peak_extra_gb"] = (torch.cuda.max_memory_allocated() - baseline) / 1e9
        result["peak_total_gb"] = torch.cuda.max_memory_allocated() / 1e9
    return result

print("BENCHMARK SETUP")

full_image = Image.open(IMAGE_PATH).convert("RGB")
print(f"image: {IMAGE_PATH}")
print(f"native size: {full_image.size}")
if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | total memory: {props.total_memory / 1e9:.1f} GB")
    print(f"model weights resident: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"warmup={N_WARMUP}, iterations={N_ITERS}")

# build the vocabulary index up front so its one-time cost is not counted in any timing
from_label_warmup = evaluate_decision(
    image=resized(full_image, 224), state=shared_state, **CASES["noul"]
)
print("warmup call ok:", from_label_warmup)


results = []

for case_name, kw in CASES.items():
    print("\n" + "=" * 78)
    print(f"Case: {case_name}")
    print("=" * 78)
    for side in SIZES:
        img = resized(full_image, side)
        size_label = "native" if side is None else str(side)

        try:
            n_tokens = count_input_tokens(img, kw["question"], kw["primitive_type"], kw["criteria"])
            stats = benchmark(lambda: evaluate_decision(image=img, state=shared_state, **kw))
        except torch.cuda.OutOfMemoryError:
            print(f"  side={size_label:>6}  size={str(img.size):>12}  OUT OF MEMORY")
            torch.cuda.empty_cache()
            continue

        results.append((case_name, size_label, img.size, n_tokens, stats))
        mem = (f" | +{stats['peak_extra_gb']:.2f} GB (total {stats['peak_total_gb']:.2f})"
               if HAS_CUDA else "")
        print(f"  side={size_label:>6}  size={str(img.size):>12}  tokens={n_tokens:>5}  "
              f"mean={stats['mean_ms']:8.1f} ms  p50={stats['p50_ms']:8.1f}  "
              f"p95={stats['p95_ms']:8.1f}  {stats['decisions_per_sec']:6.2f}/s{mem}")

# Summary tables
print("\n" + "=" * 78)
print("SUMMARY")
print("=" * 78)
header = (f"{'case':8s} {'side':>7s} {'size':>13s} {'tokens':>7s} {'mean ms':>9s} "
          f"{'p95 ms':>9s} {'std':>7s} {'dec/s':>7s}")
if HAS_CUDA:
    header += f" {'+GB':>7s} {'total GB':>9s}"
print(header)
print("-" * len(header))
for case_name, size_label, size, n_tokens, s in results:
    line = (f"{case_name:8s} {size_label:>7s} {str(size):>13s} {n_tokens:7d} "
            f"{s['mean_ms']:9.1f} {s['p95_ms']:9.1f} {s['std_ms']:7.1f} {s['decisions_per_sec']:7.2f}")
    if HAS_CUDA:
        line += f" {s['peak_extra_gb']:7.2f} {s['peak_total_gb']:9.2f}"
    print(line)

# Speedup relative to native, per case
print("\nSpeedup vs native resolution (mean latency):")
for case_name in CASES:
    native = [s for c, l, _, _, s in results if c == case_name and l == "native"]
    if not native:
        continue
    base = native[0]["mean_ms"]
    parts = [f"{l}: {base / s['mean_ms']:.1f}x"
             for c, l, _, _, s in results if c == case_name and l != "native"]
    print(f"  {case_name:7s} " + " | ".join(parts))



BENCHMARK SETUP
image: fire.jpg
native size: (1920, 1080)
GPU: Tesla T4 | total memory: 15.6 GB
model weights resident: 4.49 GB
warmup=3, iterations=20
warmup call ok: {'type': 'noul', 'noul': 0.9887999892234802, 'probabilities': {'Yes': 0.9887999892234802, 'No': 0.01119999960064888}}

Case: noul
  side=   224  size=  (224, 126)  tokens=  137  mean=   170.4 ms  p50=   163.1  p95=   216.9    5.87/s | +0.05 GB (total 4.54)
  side=   336  size=  (336, 189)  tokens=  137  mean=   171.2 ms  p50=   161.2  p95=   204.7    5.84/s | +0.05 GB (total 4.54)
  side=   448  size=  (448, 252)  tokens=  183  mean=   139.4 ms  p50=   143.2  p95=   154.8    7.17/s | +0.05 GB (total 4.54)
  side=   672  size=  (672, 378)  tokens=  323  mean=   186.5 ms  p50=   183.4  p95=   206.0    5.36/s | +0.08 GB (total 4.57)
  side=   896  size=  (896, 504)  tokens=  519  mean=   324.0 ms  p50=   316.3  p95=   365.1    3.09/s | +0.12 GB (total 4.60)
  side=  1344  size= (1344, 756)  tokens= 1079  mean=   802.7 ms  p